In [ ]:
# ============================================================
# Cell 1 — FIX 1 & 2: Working directory + sys.path
# BUG: Original used __file__ which raises NameError in Jupyter.
# BUG: src/ was never added to sys.path → ModuleNotFoundError.
# ============================================================
import os, sys

os.chdir('../')              # go up from research/ to project root
sys.path.insert(0, 'src')    # make cnnClassifier importable

print('Working dir :', os.getcwd())
print('sys.path[0] :', sys.path[0])


In [ ]:
# ============================================================
# Cell 2 — FIX 3 & 4: Imports + DataIngestionConfig
# FIX 3: Removed fragile try/except __file__ block entirely.
# FIX 4: source_URL key now matches config.yaml (both uppercase).
# ============================================================
import os
from dataclasses import dataclass
from pathlib import Path

from cnnClassifier.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from cnnClassifier.utils.common import read_yaml, create_directories


@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str        # matches config.yaml key 'source_URL'
    local_data_file: Path
    unzip_dir: Path


In [ ]:
# ============================================================
# Cell 3 — FIX 5: ConfigurationManager
# Replaced the 15-line fragile key-fallback loop with a single
# direct attribute access: config.source_URL
# ============================================================
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=PARAMS_FILE_PATH,
    ):
        self.config = read_yaml(Path(config_filepath))
        self.params = read_yaml(Path(params_filepath))
        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        # FIX 5: simple direct access — no fragile fallback loop
        return DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir),
        )


In [ ]:
# ============================================================
# Cell 4 — FIX 6: DataIngestion class (was ENTIRELY MISSING)
# Added download_file() with gdown fuzzy=True
# Added extract_zip_file() with zipfile
# ============================================================
import zipfile
import gdown


class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self):
        """Download zip from Google Drive using gdown."""
        if not os.path.exists(self.config.local_data_file):
            # fuzzy=True handles /view?usp=sharing and folder-style links
            gdown.download(
                self.config.source_URL,
                str(self.config.local_data_file),
                quiet=False,
                fuzzy=True,
            )
            print(f'Downloaded: {self.config.local_data_file}')
        else:
            size = os.path.getsize(self.config.local_data_file)
            print(f'File already exists ({size} bytes) — skipping download.')

    def extract_zip_file(self):
        """Extract the downloaded zip into unzip_dir."""
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as z:
            z.extractall(unzip_path)
        print(f'Extracted to: {unzip_path}')


In [ ]:
# ============================================================
# Cell 5 — Run pipeline (download from Google Drive)
# ============================================================
config_manager = ConfigurationManager()
data_ingestion_config = config_manager.get_data_ingestion_config()

data_ingestion = DataIngestion(config=data_ingestion_config)
data_ingestion.download_file()
data_ingestion.extract_zip_file()

print('\n✅ Data Ingestion Complete!')


In [ ]:
# ============================================================
# Cell 6 — SHORTCUT ⚡ (recommended — run instead of Cell 5)
# Kidney-CT-Scan-data.zip already exists in research/ folder!
# Skip Google Drive download — copy + extract locally.
# ============================================================
import shutil, zipfile, os

local_zip  = 'research/Kidney-CT-Scan-data.zip'  # already present!
dest_zip   = 'artifacts/data_ingestion/data.zip'
extract_to = 'artifacts/data_ingestion'

os.makedirs('artifacts/data_ingestion', exist_ok=True)

if os.path.exists(local_zip):
    shutil.copy(local_zip, dest_zip)
    print(f'Copied  : {local_zip} -> {dest_zip}')
    with zipfile.ZipFile(dest_zip, 'r') as z:
        z.extractall(extract_to)
    print(f'Extracted: {extract_to}')
    print('\n✅ Data Ingestion Complete (from local zip)!')
else:
    print('Local zip not found — run Cell 5 to download from Google Drive.')
